In [4]:
import pandas as pd

print("Loading Accident Database...")
# Load the accident data to find which drivers actually had crashes
df_accidents = pd.read_excel('../Data/Traffic Accident Database 2024.xlsx')
# Create a set of employee numbers for fast lookup
at_risk_drivers = set(df_accidents['Employee No'].dropna().unique())

def load_all_sheets(filepath):
    print(f"Reading sheets from {filepath}...")
    xls = pd.ExcelFile(filepath)
    df_list = []
    for sheet in xls.sheet_names:
        print(f" - Loading sheet: {sheet}")
        df_sheet = pd.read_excel(xls, sheet_name=sheet)
        df_list.append(df_sheet)
    return pd.concat(df_list, ignore_index=True)

# Load the split alarm files
df_part1 = load_all_sheets('../Data/Mar24_part1.xlsx')
df_part2 = load_all_sheets('../Data/Mar24_part2.xlsx')

# Combine all March data into a single DataFrame
df_alarms = pd.concat([df_part1, df_part2], ignore_index=True)
print(f"\nTotal alarm records loaded: {len(df_alarms):,}")

# Map the target variable: 1 if the driver is in the accident database, 0 otherwise
df_alarms['At_Risk'] = df_alarms['Driver'].apply(lambda x: 1 if x in at_risk_drivers else 0)

# Extract the first 1000 rows for initial exploration as suggested in the project briefing
df_exploration = df_alarms.head(1000)
df_exploration.to_csv("../Output/CEG2003_Exploration_1000.csv", index=False)
print("\nSaved first 1000 rows to '../Output/CEG2003_Exploration_1000.csv' for quick review.")

Loading Accident Database...
Reading sheets from ../Data/Mar24_part1.xlsx...
 - Loading sheet: 1_5Mar_24
 - Loading sheet: 6_10Mar_24
 - Loading sheet: 11_15Mar_24
Reading sheets from ../Data/Mar24_part2.xlsx...
 - Loading sheet: 16_20Mar_24
 - Loading sheet: 21_25Mar_24
 - Loading sheet: 26_31Mar_24

Total alarm records loaded: 4,167,176

Saved first 1000 rows to '../Output/CEG2003_Exploration_1000.csv' for quick review.


In [5]:
import pandas as pd

# 1. Pivot categorical alarm types into frequency counts per driver
# This creates columns like 'Fatigue alert', 'Harsh Braking', etc., filled with counts
alarm_counts = pd.crosstab(df_alarms['Driver'], df_alarms['Alarm type'])

# Optional: Rename columns to be more machine-learning friendly (remove spaces)
alarm_counts.columns = [f"Count_{col.replace(' ', '_')}" for col in alarm_counts.columns]

# 2. Aggregate continuous numerical columns per driver
numerical_aggs = df_alarms.groupby('Driver').agg(
    mean_speed=('speed(km/h)', 'mean'),
    max_speed=('speed(km/h)', 'max'),
    total_alarm_duration=('Alarm duration', 'sum'),
    mean_alarm_duration=('Alarm duration', 'mean'),
    total_alarms_triggered=('Alarm type', 'count')
)

# 3. Combine both dataframes into the final Feature Matrix (X)
driver_features = alarm_counts.join(numerical_aggs).reset_index()

# Display the first few rows of the new engineered features
print(driver_features.head())

# 4. Map the 'At_Risk' target variable to the aggregated driver features
# Extract the unique Driver and At_Risk pairs from your original df_alarms
risk_mapping = df_alarms[['Driver', 'At_Risk']].drop_duplicates()

# Merge the risk labels into your engineered features
final_dataset = pd.merge(driver_features, risk_mapping, on='Driver', how='left')

# Fill any potential NaN values with 0 (just in case)
final_dataset['At_Risk'] = final_dataset['At_Risk'].fillna(0).astype(int)

# 5. Export the final dataset to CSV
final_dataset.to_csv("../Output/driver_features.csv", index=False)
print(f"Successfully exported {len(final_dataset)} drivers to '../Output/driver_features.csv'")

    Driver  Count_Fatigue_Alert  Count_Forward_Collision_Warning  \
0  10055.0                    2                              512   
1  10074.0                   93                             1169   
2  10102.0                  185                              108   
3  10138.0                    1                              267   
4  10151.0                    1                                7   

   Count_Harsh_Acceleration  Count_Harsh_Braking  \
0                       709                 1864   
1                       138                 2028   
2                        24                  325   
3                       129                 1227   
4                         1                   23   

   Count_Headway_Monitoring_Alert  Count_Pedestrian_Collision_Alert  \
0                               2                               104   
1                             170                               105   
2                              40                                